# Carderno 5 - Gerar embeddings dos documentos e queries

In [18]:
import json
from openai import OpenAI
import os
from tqdm import tqdm
from getpass import getpass
import h5py
import numpy as np
from formatador import html_to_plain_text
import re

In [2]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'

ARQUIVO_DOCS = f'{PASTA_DOCS_QRELS}/docs.csv'
ARQUIVO_QUERIES = f'{PASTA_DOCS_QRELS}/query.csv'
ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}/qrel.csv'

In [3]:
OPENAI_KEY = getpass("KEY OpenAI")
NOME_MODELO_EMB_OPENAI_LARGE = "text-embedding-3-large"
DIM_MODELO_EMB_OPENAI_LARGE = 3072

# Modelos disponíveis
MODELOS_EMB_NOME_E_DIM_EMB = [(NOME_MODELO_EMB_OPENAI_LARGE, DIM_MODELO_EMB_OPENAI_LARGE)]

# Modelos para gerar. A ideia é que, uma vez que já foi gerado, pode tirar daqui. Daí ele não precisa carregar o arquivo e ver se está lá
MODELOS_EMB_PARA_GERAR = [(NOME_MODELO_EMB_OPENAI_LARGE, DIM_MODELO_EMB_OPENAI_LARGE)]
MODELOS_EMB_PARA_GERAR = []

KEY OpenAI ········


Os embeddings serão gerados no arquivo abaixo. No entanto, depois vou copiar um conjunto de embeddings por arquivo para subir pro git.

In [4]:
ARQUIVO_EMBEDDINGS_DOCS = './dados/outputs/5 - embeddings/embeddings_docs.h5'
ARQUIVO_EMBEDDINGS_QUERIES = './dados/outputs/5 - embeddings/embeddings_queries.h5'

# 1. Carregar a base de documentos e de queries

In [69]:
import pandas as pd

docs = pd.read_csv(ARQUIVO_DOCS)
queries = pd.read_csv(ARQUIVO_QUERIES)

docs['TEXTONORMA'] = docs['TEXTONORMA'].fillna('')
docs['ASSUNTO'] = docs['ASSUNTO'].fillna('')

In [6]:
print(docs.columns)
print(queries.columns)

Index(['KEY', 'UNIDADEBASICAAUTORA', 'ORIGEM', 'NUMNORMA', 'ANONORMA',
       'TIPONORMA', 'NUMEROPROCESSO', 'NUMEROPROCESSOFORMATADO', 'TITULO',
       'ASSUNTO', 'TEXTONORMA', 'DATAINICIOVIGENCIA', 'DATAFIMVIGENCIA',
       'SITUACAO', 'LINKBTCU', 'TEXTOANEXO', 'ARQUIVONORMA', 'PAGINABTCU',
       'TEMA', 'TAGSVCE', 'NORMARELACIONADA', 'NUMDOU', 'NUMSECAODOU',
       'NUMPAGINADOU', 'DATADOU', 'INFOSGERAIS'],
      dtype='object')
Index(['KEY', 'TEXT'], dtype='object')


# 2. Criar as estruturas em arquivos H5

In [7]:
# Cria uma estrutura h5. A estrutura é salva no arquivo 'arquivo'.
# A ideia da estrutura é a seguinte:
#   - Um dataset contendo a lista de ids que será salvo. 
#       O nome desse dataset poderia ser padrão (por exemplo, ID) já que será um h5 para documentos e outro para as queries. No entanto,
#       usarei DOC_KEY e QUERY_KEY simplesmente pq é o vocabulário que estamos usando para docs e queries. Esse dataset já é inicializado
#       com a lista de todas as ids.
#   - Um dataset por modelo de embeddings. Nesse caso, o nome do dataset é o nome do modelo de embeddings e, quando é criado, é criado
#       sem embeddings (serão gerados depois). A ideia é que os embeddings são pareados com as ids, ou seja, o embeddings na posição i
#       se refere ao i'éssima id.
def criar_estrutura_embeddings(arquivo, nome_id, lista_id):
    with h5py.File(arquivo, "a") as f:
        # Tamanho do dataset
        n = len(lista_id)
        # Tamanho dos chunks para gravar no dataset
        chunk_size = min(128, n)
        
        # Cria o dataset para as IDs dos docs/queries
        if nome_id not in f:
            f.create_dataset(
                nome_id,
                data=np.array(lista_id, dtype="S"),  # grava tudo de uma vez
                maxshape=(None,),
                dtype=h5py.string_dtype(encoding="utf-8"),
                chunks=True
            )

        # Datasets para os embeddings
        for nome_modelo, dim_modelo in MODELOS_EMB_NOME_E_DIM_EMB:
            if nome_modelo not in f:
                # Quando criar o dataset com os embeddings do modelo, cria preenchido com nan
                ds = f.create_dataset(
                    nome_modelo,
                    shape=(n, dim_modelo),
                    maxshape=(n, dim_modelo),
                    dtype=np.float16,
                    compression="gzip",
                    chunks=(chunk_size, dim_modelo)
                )
                ds[:] = np.nan

criar_estrutura_embeddings(ARQUIVO_EMBEDDINGS_DOCS, 'DOC_KEY', docs["KEY"].tolist())
criar_estrutura_embeddings(ARQUIVO_EMBEDDINGS_QUERIES, 'QUERY_KEY', queries["KEY"].tolist())

Funções auxiliares para saber se já existe embedding associado e para atualizar embeddings:

In [8]:
def existe_embedding(arquivo, nome_modelo, idx):
    with h5py.File(arquivo, "a") as f:
        ds = f[nome_modelo]

        return not np.isnan(ds[idx, 0])
    
def atualizar_embedding(arquivo, nome_modelo, idx, embedding):
    with h5py.File(arquivo, "a") as f:
        ds = f[nome_modelo]

        ds[idx] = np.asarray(embedding, dtype=np.float16)

## 3. Cria embeddings

Funções auxiliares para geração de embeddings

In [22]:
def reduz_texto_entrada(texto, nome_modelo, exc: Exception):
    error_text = str(exc)

    if nome_modelo == 'text-embedding-3-large':
        match = re.search(r'requested\s+(\d+)\s+tokens', exc.message)
        
        total_token_requisitados = int(match.group(1))
        perc_reduzir = 8192/total_token_requisitados * .95
        texto = texto[:int(len(texto) * perc_reduzir)]
        
        return texto

In [27]:
# A id só é usada para fazer o log em caso de erro
def extrai_emb_com_retry(id, texto, nome_modelo, func_get_emb):
    try:
        return func_get_emb(nome_modelo, texto)
    except Exception as e:
        #tqdm.write(f'id: {id}. Texto muito grande. Reduzindo o tamanho')
        return extrai_emb_com_retry(id, reduz_texto_entrada(texto, nome_modelo, e), nome_modelo, func_get_emb)

Função para extrair embeddings para os modelos da openAI:

In [11]:
client_openai = OpenAI(api_key=OPENAI_KEY, base_url=None)
def extrair_embeddings_openai(nome_modelo, texto):
   return client_openai.embeddings.create(input = [texto], model=nome_modelo).data[0].embedding

In [12]:
mapa_func_extrair_embeddings = {
    NOME_MODELO_EMB_OPENAI_LARGE: extrair_embeddings_openai
}

In [61]:
texto

nan

In [62]:
row

KEY                                                               NORMA-4848
UNIDADEBASICAAUTORA                                                [SEGEDAM]
ORIGEM                                                               [SEGEP]
NUMNORMA                                                                  46
ANONORMA                                                                2014
TIPONORMA                                                           Portaria
NUMEROPROCESSO                                                           NaN
NUMEROPROCESSOFORMATADO                                                  NaN
TITULO                                             Portaria SEGEP n° 46/2014
ASSUNTO                                                                  NaN
TEXTONORMA                                                                  
DATAINICIOVIGENCIA                                                11/08/2014
DATAFIMVIGENCIA                                                          NaN

Agora gera os embeddings do texto da norma:

In [71]:
# Varre todos os textos das normas
for idx, row in tqdm(docs.iterrows(), total=len(docs)):
    doc_key = row['KEY']
    # Tem uma norma com mais de 1milhão de caracteres que extrapola até o limite de tokens de entrada por minuto.
    # (mais de 300k, mesmo o limite da janela de entrada sendo muito menor que isso - 8k pros modelos da OpenAI).
    texto = html_to_plain_text(row['TEXTONORMA'])[:1_000_000]
    # Algumas normas tem o texto vazio. Então considera o assunto ou até mesmo a key, apenas para ter um conjunto de emb indexado
    # Por exemplo: NORMA-7114, NORMA-4885, NORMA-4848...
    if texto == '':
        texto = row['ASSUNTO']
        print(f'Texto da norma {doc_key} vazio')
    if texto == '':
        texto = row['KEY']
        print(f'Assunto da norma {doc_key} vazio')

    for nome_modelo, _ in MODELOS_EMB_PARA_GERAR:
        if not existe_embedding(ARQUIVO_EMBEDDINGS_DOCS, nome_modelo, idx):
            func_extrair_emb = mapa_func_extrair_embeddings[nome_modelo]
            emb_gerado = extrai_emb_com_retry(doc_key, texto, nome_modelo, func_extrair_emb)
            atualizar_embedding(ARQUIVO_EMBEDDINGS_DOCS, nome_modelo, idx, emb_gerado)

 55%|█████████████████████████████████████████▍                                  | 7887/14469 [00:52<00:42, 155.54it/s]

Texto da norma NORMA-7114 vazio


 61%|██████████████████████████████████████████████▌                             | 8876/14469 [00:58<00:41, 134.51it/s]

Texto da norma NORMA-4885 vazio
Assunto da norma NORMA-4885 vazio


 64%|████████████████████████████████████████████████▌                           | 9247/14469 [01:00<00:29, 175.26it/s]

...

 86%|█████████████████████████████████████████████████████████████████▌          | 12472/14469 [25:07<19:34,  1.70it/s]

Texto da norma NORMA-18218 vazio
Assunto da norma NORMA-18218 vazio


 94%|███████████████████████████████████████████████████████████████████████     | 13530/14469 [33:37<06:01,  2.60it/s]

Texto da norma NORMA-464 vazio


100%|████████████████████████████████████████████████████████████████████████████| 14469/14469 [40:55<00:00,  5.89it/s]


Gera os embeddings das queries:

In [72]:
# Varre todos os enunciados das queries
for idx, row in tqdm(queries.iterrows(), total=len(queries)):
    texto = row['TEXT']

    for nome_modelo, _ in MODELOS_EMB_PARA_GERAR:
        if not existe_embedding(ARQUIVO_EMBEDDINGS_QUERIES, nome_modelo, idx):
            func_extrair_emb = mapa_func_extrair_embeddings[nome_modelo]
            emb_gerado = extrai_emb_com_retry(id, texto, nome_modelo, func_extrair_emb)
            atualizar_embedding(ARQUIVO_EMBEDDINGS_QUERIES, nome_modelo, idx, emb_gerado)

100%|██████████████████████████████████████████████████████████████████████████████████| 46/46 [00:17<00:00,  2.68it/s]
